# Accessing data

In [1]:
#This gets us into the right directory from home, in order to run the import python script from Mat

#sys = system (module), gives information and control over python interpreter/terminal itself
import sys
sys.path.append("home/565/pv3484/aus_substation_electricity")

#% is a magic command, special shortcut command that lets you control/interact with notebook environment (ie. gives control of terminal without writing full python code)
%cd aus_substation_electricity/

!pwd

/home/565/pv3484/aus_substation_electricity
/home/565/pv3484/aus_substation_electricity


In [2]:
#This section imports the substations that Mat put together

%run /home/565/pv3484/aus_substation_electricity/import_substation.py

processing nsw substations for ['ausgrid'] from None to None
ausgrid
following columns in demand are not in info index:
['MT_HU', 'SI_NO']
removing these columns from demand
number of substations in ausgrid substation info: 134
number of substations in ausgrid substation data: 132
following sites match selection criteria:
               energy_asset          Name  Area  Dwellings  Persons  Residential  Commercial  Industrial  Primary Production  Education  \
ID                                                                                                                                        
BLAKE         AG_BLAKEHURST    Blakehurst     7      10081    28521        0.850       0.005       0.021               0.000      0.022   
PUNCH          AG_PUNCHBOWL     Punchbowl     9      17514    50395        0.826       0.048       0.038               0.000      0.025   
MEADO         AG_MEADOWBANK    Meadowbank    15      22420    56948        0.825       0.023       0.015               0

## Holiday Function

In [15]:
import pandas as pd
from datetime import date, timedelta
from dateutil.easter import easter

In [16]:
# Define all national public holidays (including moving ones like Easter)
HOLIDAYS_VIC = {
    "New Year's Day": lambda y: pd.Timestamp(f"{y}-01-01"),
    "Australia Day": lambda y: pd.Timestamp(f"{y}-01-26"),
    "Good Friday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=2),
    "Easter Saturday": lambda y: pd.Timestamp(easter(y)) - pd.Timedelta(days=1),
    "Easter Sunday": lambda y: pd.Timestamp(easter(y)),
    "Easter Monday": lambda y: pd.Timestamp(easter(y)) + pd.Timedelta(days=1),
    "ANZAC Day": lambda y: pd.Timestamp(f"{y}-04-25"),
    "Christmas Day": lambda y: pd.Timestamp(f"{y}-12-25"),
    "Boxing Day": lambda y: pd.Timestamp(f"{y}-12-26"),
}

# Putting confidence interval shading on the anomaly plots
- Using the existing code from 'anomaly_stratify_hour.ipynb'
- Adding in confidence interval shading

In [17]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

def compare_holiday_only_window(
    demand,
    info,
    station,
    years,                 # must be a 2‑tuple: (start_year, end_year)
    holiday_func,
    holiday_name,
):
    """
    Compute and plot holiday demand anomalies using a ±30‑day window
    around the holiday for a *2‑year baseline window*.

    Returns
    -------
    fig : matplotlib.figure.Figure
        The figure object (not shown, not saved).
    """

    # --- Validate input ---
    if not (isinstance(years, tuple) and len(years) == 2):
        raise ValueError("`years` must be a tuple like (2004, 2005).")

    start_year, end_year = years
    year_list = list(range(start_year, end_year + 1))

    # --- Prepare hourly demand data ---
    demand.index = pd.to_datetime(demand.index)
    hourly = demand[[station]].resample("h").mean()

    # --- Collect ±30‑day windows around the holiday for each year ---
    windows = []
    for yr in year_list:
        ref_date = holiday_func(yr)
        start = ref_date - pd.Timedelta(days=30)
        end   = ref_date + pd.Timedelta(days=30)
        windows.append(hourly.loc[start:end].copy())

    combined = pd.concat(windows)

    # --- Compute baseline stratified by hour ---
    combined["hour"] = combined.index.hour
    baseline_by_hour = combined.groupby("hour")[station].mean()

    # --- Compute anomalies ---
    anomalies = combined.copy()
    anomalies["anomaly"] = anomalies[station] - anomalies["hour"].map(baseline_by_hour)

    # --- Extract 24‑hour anomaly profile for each year ---
    holiday_hours = []
    for yr in year_list:
        ref_date = holiday_func(yr)
        expected_hours = pd.date_range(ref_date, ref_date + pd.Timedelta(hours=23), freq="h")

        daily = anomalies["anomaly"].reindex(expected_hours)
        daily = daily.interpolate(limit_direction="both")  # fill small gaps
        daily.index = range(len(daily))
        daily = daily.reindex(range(24))

        holiday_hours.append(daily)

    # --- Build matrix of per‑year profiles ---
    holiday_matrix = pd.concat(holiday_hours, axis=1)

    # --- Mean + CI ---
    holiday_profile = holiday_matrix.mean(axis=1)

    valid_years = holiday_matrix.count(axis=0)
    holiday_matrix = holiday_matrix.loc[:, valid_years > 0]

    if holiday_matrix.shape[1] < 2:
        ci95 = pd.Series([np.nan] * 24)
    else:
        std_profile = holiday_matrix.std(axis=1)
        n_years = holiday_matrix.count(axis=1)
        ci95 = 1.96 * std_profile / np.sqrt(n_years)

    # --- Plot ---
    fig, ax = plt.subplots(figsize=(12, 5))

    ax.fill_between(
        holiday_profile.index,
        holiday_profile - ci95,
        holiday_profile + ci95,
        where=~ci95.isna(),
        color="red",
        alpha=0.2,
        label="95% CI"
    )

    ax.plot(
        holiday_profile.index,
        holiday_profile.values,
        color="red",
        linewidth=2,
        marker="o",
        label=f"{holiday_name} (±30‑Day Window)"
    )

    ax.axhline(0, color="black", linewidth=1)
    ax.grid(axis='y', linestyle='-', linewidth=0.5, color='gray', alpha=0.3)
    ax.set_xticks(np.arange(0, 24))
    ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45)

    full_name = info.loc[station, "Name"]
    ax.set_title(
        f"{full_name} Demand Anomaly: {holiday_name} ({start_year}–{end_year}, ±30‑Day Window)",
        fontsize=14
    )
    ax.set_xlabel("Hour of Day")
    ax.set_ylabel("Electricity Demand Anomaly")
    ax.legend()
    fig.tight_layout()

    # --- IMPORTANT: do NOT show the plot ---
    plt.close(fig)

    return fig

## Testing function with just BLAKE

In [ ]:
compare_holiday_only_window(
    demand=demand,
    info=info,
    station="BLAKE",
    years=[2012, 2013],
    holiday_func=lambda y: pd.Timestamp(y, 12, 25),
    holiday_name="Christmas Day"
)

## Looping through all holidays for all substations
- Saves each substation in a separate folder
- Creates a pdf of each holiday containing every 2 year interval with 95% confidence interval

In [5]:
#Build rolloing 2 year window
year_windows = [(y, y+1) for y in range(2004, 2018)]

In [ ]:
import os
from matplotlib.backends.backend_pdf import PdfPages

BASE_DIR = "/home/565/pv3484/aus_substation_electricity/figures/anomaly_demand/confidence_interval"

year_windows = [(y, y+1) for y in range(2004, 2018)]

for station in demand.columns:
    
    station_dir = os.path.join(BASE_DIR, station)
    os.makedirs(station_dir, exist_ok=True)

    for holiday_name, holiday_func in HOLIDAYS_VIC.items():

        pdf_path = os.path.join(station_dir, f"{station}_{holiday_name}.pdf")

        with PdfPages(pdf_path) as pdf:

            for start_year, end_year in year_windows:

                fig = compare_holiday_only_window(
                    demand=demand,
                    info=info,
                    station=station,
                    years=(start_year, end_year),
                    holiday_func=holiday_func,
                    holiday_name=holiday_name,
                )

                pdf.savefig(fig)
                plt.close(fig)